- https://thinkingmachines.ai/blog/defeating-nondeterminism-in-llm-inference/
    - https://github.com/thinking-machines-lab/batch_invariant_ops.git

In [1]:
import torch

In [3]:
A = torch.randn(2048, 2048, device='cuda', dtype=torch.bfloat16)
B = torch.randn(2048, 2048, device='cuda', dtype=torch.bfloat16)
ref = torch.mm(A, B)
for _ in range(1000):
    assert (torch.mm(A, B) - ref).abs().max().item() == 0

### GPUs

- atomic adds
    - GPU launches a program concurrently across many “cores” (i.e. SMs). As the cores have no inherent synchronization among them, this poses a challenge if the cores need to communicate among each other.
    - The atomic add is “nondeterministic” — the order in which the results accumulate is purely dependent on which core finishes first.
    - In fact, in the typical forward pass of an LLM, there is usually not a single atomic add present.

In [4]:
torch.set_default_device('cuda') 

for i in range(10):
    B = 2048
    D = 4096
    a = torch.linspace(-1000, 1000, B*D).reshape(B, D)
    b = torch.linspace(-1000, 1000, D*D).reshape(D, D)
    # Doing a matrix vector multiplication by taking
    # the first element of the batch
    out1 = torch.mm(a[:1], b)
    # Doing a matrix matrix multiplication and then taking
    # the first element of the batch
    out2 = torch.mm(a, b)[:1]
    print((out1 - out2).abs().max()) # tensor(1669.2500, device='cuda:0')

tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')
tensor(1152., device='cuda:0')


### highlights

- 之前社区普遍认为，这是因为 GPU 上的浮点运算不满足结合律（(a + b) + c ≠ a + (b + c)），加上并发执行时线程完成顺序的不确定性，导致最终计算结果不同。	
    - 这篇blog的结论：The requirement for batch invariance is that the **reduction order for each element must be fixed** regardless of the batch-size of the kernel.
        - we only break batch invariance when our batch-size affects the reduction strategy.
    - 内核对批大小/切分的敏感而非单纯的“并发+原子加”（ concurrency and atomic adds）
        - 解决方案
            - batch-invariant matmul 思路（固定内核配置、禁用 Split-K/stream-k、固定瓦片/指令族）；
- 真正“同策略”的强化学习（True On-Policy RL）：在强化学习中，采样（推理）和训练过程的数值差异会导致原本的“同策略”算法变成事实上的“异策略”，可能导致训练崩溃。通过确保采样器和训练器之间的结果比特一致，该技术可以实现真正的“同策略”训练，从而提升训练的稳定性和效果。
- we only need to worry about the 3 operations that involve reductions 
    - RMSNorm: 要对一个向量求均值/方差或平方和，再开方、归一化。
    - matrix multiplication：$y_i = ∑_k A_{ik}·B_{kj}$
    - and attention: softmax 的分母 $∑_j \exp(\text{score}_j)$ 是沿序列维的求和

### codes

- batch_invariant_ops

#### 矩阵乘法

```python
def test_batch_invariance(): 
    B, D = 2048, 4096 
    a = torch.linspace(-100, 100, B*D).reshape(B, D) 
    b = torch.linspace(-100, 100, D*D).reshape(D, D) 
    # Method 1: Matrix-vector multiplication (batch size 1) 
    out1 = torch.mm(a[:1], b) 
    # Method 2: Matrix-matrix multiplication, then slice (full batch) 
    out2 = torch.mm(a, b)[:1] 
    # Check if results are identical 
    diff = (out1 - out2).abs().max() 
    print(f"Difference: {diff.item()}") 
    return diff.item() == 0
```

```
Standard PyTorch:
Difference: 4.1953125
Deterministic: False

Batch-Invariant Mode:
Difference: 0.0
Deterministic: True
```